# Capstone 09 - Day 1: Bronze ingestion (Auto Loader)

Loads the ADLS event files from `bronze.raw_volume` into append-only Bronze tables (raw STRING values, no dedup), profiles them and runs the Day 1 tests. Grain: one event row per `reading_id` per source arrival. Reference entities are already in `bronze_meter_id` / `bronze_substation_id`.

In [0]:
from pyspark.sql import functions as F

CATALOG, SCHEMA = "capstone9_smartmeter", "bronze"
SRC  = f"/Volumes/{CATALOG}/{SCHEMA}/raw_volume/adls"   # source files (read-only)
META = f"/Volumes/{CATALOG}/{SCHEMA}/pipeline_meta"     # Auto Loader schema + checkpoint locations
T = {s: f"{CATALOG}.{SCHEMA}.{s}_raw" for s in ["primary_events", "secondary_events", "status_updates"]}
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.pipeline_meta")

def ingest(src):
    (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv").option("header", "true")
        .option("cloudFiles.inferColumnTypes", "false")                # keep raw values as STRING
        .option("cloudFiles.schemaLocation", f"{META}/_schemas/{src}")
        .load(f"{SRC}/{src}/")
        .select("*", F.col("_metadata.file_path").alias("_input_file_name"), F.current_timestamp().alias("_ingestion_timestamp"))
        .writeStream.option("checkpointLocation", f"{META}/_checkpoints/{src}")
        .option("mergeSchema", "true").trigger(availableNow=True)
        .toTable(T[src]).awaitTermination())
    return spark.table(T[src]).count()

In [0]:
print({s: ingest(s) for s in T})

In [0]:
%sql
SELECT count(*)                                         AS row_count,
       count(DISTINCT reading_id)                       AS distinct_reading_id,
       count(*) - (SELECT count(*) FROM (SELECT DISTINCT reading_id, event_timestamp, ingest_timestamp, substation_id, meter_id, tariff_type,
                   meter_status, unit, consumption_kwh, voltage_v, feeder_id, source_system FROM capstone9_smartmeter.bronze.primary_events_raw)) AS exact_duplicates,
       count_if(tariff_type IS NULL)                    AS null_tariff_type,
       count_if(try_cast(consumption_kwh AS DOUBLE) > 12) AS extreme_kwh_gt_12,
       count_if(try_cast(consumption_kwh AS DOUBLE) IS NULL OR try_cast(voltage_v AS DOUBLE) IS NULL
                OR try_to_timestamp(event_timestamp) IS NULL OR try_to_timestamp(ingest_timestamp) IS NULL) AS type_cast_failures,
       count_if(source_schema_version IS NOT NULL)      AS v2_schema_drift_rows,
       count(DISTINCT _input_file_name)                     AS files
FROM capstone9_smartmeter.bronze.primary_events_raw

In [0]:
%sql
WITH sec AS (SELECT * FROM capstone9_smartmeter.bronze.secondary_events_raw),
     st  AS (SELECT * FROM capstone9_smartmeter.bronze.status_updates_raw),
     m   AS (SELECT * FROM capstone9_smartmeter.bronze.bronze_meter_id),
     sb  AS (SELECT * FROM capstone9_smartmeter.bronze.bronze_substation_id)
SELECT 'secondary_events' AS source, count(*) AS row_count, count(DISTINCT event_id) AS distinct_key,
       count(*) - (SELECT count(*) FROM (SELECT DISTINCT event_id, reading_id, event_timestamp, related_entity_id, event_type, event_value, source_system FROM sec)) AS exact_duplicates,
       count_if(event_id IS NULL OR reading_id IS NULL OR event_timestamp IS NULL OR related_entity_id IS NULL OR event_type IS NULL OR event_value IS NULL OR source_system IS NULL) AS rows_with_nulls,
       count_if(try_to_timestamp(event_timestamp) IS NULL OR try_cast(event_value AS DOUBLE) IS NULL) AS type_cast_failures
FROM sec
UNION ALL
SELECT 'status_updates', count(*), count(DISTINCT reading_id),
       count(*) - (SELECT count(*) FROM (SELECT DISTINCT reading_id, change_timestamp, previous_status, new_status, change_reason, source_sequence, source_system FROM st)),
       count_if(reading_id IS NULL OR change_timestamp IS NULL OR previous_status IS NULL OR new_status IS NULL OR change_reason IS NULL OR source_sequence IS NULL OR source_system IS NULL),
       count_if(try_to_timestamp(change_timestamp) IS NULL OR try_cast(source_sequence AS INT) IS NULL)
FROM st
UNION ALL
SELECT 'bronze_meter_id', count(*), count(DISTINCT meter_id),
       count(*) - (SELECT count(*) FROM (SELECT DISTINCT meter_id, display_name, region, active_flag, effective_date, risk_tier FROM m)),
       count_if(meter_id IS NULL OR display_name IS NULL OR region IS NULL OR active_flag IS NULL OR effective_date IS NULL OR risk_tier IS NULL),
       count_if(_rescued_data IS NOT NULL)
FROM m
UNION ALL
SELECT 'bronze_substation_id', count(*), count(DISTINCT substation_id),
       count(*) - (SELECT count(*) FROM (SELECT DISTINCT substation_id, display_name, region, active_flag, effective_date, risk_tier FROM sb)),
       count_if(substation_id IS NULL OR display_name IS NULL OR region IS NULL OR active_flag IS NULL OR effective_date IS NULL OR risk_tier IS NULL),
       count_if(_rescued_data IS NOT NULL)
FROM sb

In [0]:
%sql
SELECT 'primary_events.meter_status' AS field, meter_status AS status, count(*) AS rows FROM capstone9_smartmeter.bronze.primary_events_raw GROUP BY 2
UNION ALL
SELECT 'status_updates.new_status', new_status, count(*) FROM capstone9_smartmeter.bronze.status_updates_raw GROUP BY 2
ORDER BY 1, 2

In [0]:
%sql
WITH p AS (SELECT * FROM capstone9_smartmeter.bronze.primary_events_raw),
     m AS (SELECT meter_id FROM capstone9_smartmeter.bronze.bronze_meter_id),
     s AS (SELECT substation_id FROM capstone9_smartmeter.bronze.bronze_substation_id)
SELECT 'primary.meter_id -> bronze_meter_id' AS key_check, count(*) AS orphans FROM p LEFT ANTI JOIN m USING (meter_id)
UNION ALL SELECT 'primary.substation_id -> bronze_substation_id', count(*) FROM p LEFT ANTI JOIN s USING (substation_id)
UNION ALL SELECT 'secondary.reading_id -> primary', count(*) FROM capstone9_smartmeter.bronze.secondary_events_raw x LEFT ANTI JOIN p USING (reading_id)
UNION ALL SELECT 'secondary.related_entity_id -> bronze_substation_id', count(*) FROM capstone9_smartmeter.bronze.secondary_events_raw x LEFT ANTI JOIN s ON x.related_entity_id = s.substation_id
UNION ALL SELECT 'status_updates.reading_id -> primary', count(*) FROM capstone9_smartmeter.bronze.status_updates_raw x LEFT ANTI JOIN p USING (reading_id)

In [0]:
pe = spark.table(T["primary_events"])
biz = [c for c in pe.columns if not c.startswith("_") and c != "source_schema_version"]
before = {s: spark.table(t).count() for s, t in T.items()}
after = {s: ingest(s) for s in T}   # replay: no new files -> no new rows

tests = {
    "TC01 rows = 11,540":                 pe.count() == 11540,
    "TC01 exact duplicates = 40":         pe.count() - pe.select(biz).distinct().count() == 40,
    "TC01 distinct reading_id = 11,500":  pe.select("reading_id").distinct().count() == 11500,
    "TC02 null tariff_type = 12":         pe.filter("tariff_type IS NULL").count() == 12,
    "TC02 consumption_kwh > 12 = 35":     pe.filter("try_cast(consumption_kwh AS DOUBLE) > 12").count() == 35,
    "Replay adds no rows":                before == after,
    "Every row has _input_file_name and _ingestion_timestamp":
        all(spark.table(t).filter("_input_file_name IS NULL OR _ingestion_timestamp IS NULL").count() == 0 for t in T.values()),
}
for name, ok in tests.items(): print("PASS" if ok else "FAIL", "|", name)
assert all(tests.values()), "Day 1 tests failed"